# Generators in Python

My learning notes on **generators**: what they are, how `yield` works, and how they connect to iterators and comprehensions.

## Table of Contents
1. [Warm-up: functions and function calls](#1.-Warm-up:-functions-and-function-calls)
2. [What is a generator?](#2.-What-is-a-generator?)
3. [Your first generator](#3.-Your-first-generator)
4. [How a generator pauses and resumes](#4.-How-a-generator-pauses-and-resumes)
5. [Generators with `for` loops](#5.-Generators-with-for-loops)
6. [Generator that wraps a list](#6.-Generator-that-wraps-a-list)
7. [Comprehensions recap: list, set, generator expression](#7.-Comprehensions-recap:-list,-set,-generator-expression)
8. [Iterable vs Iterator, and `iter()`](#8.-Iterable-vs-Iterator,-and-iter())
9. [Practice problems](#9.-Practice-problems)
10. [Why generators? Memory efficiency](#10.-Why-generators?-Memory-efficiency)
11. [Summary / cheat sheet](#11.-Summary-/-cheat-sheet)

## 1. Warm-up: functions and function calls

Before generators, one small but important idea: **calling a function** and **referring to a function** are different things.

- `test()` → *runs* the function and gives back its return value
- `test` → just the function *object* (nothing runs)

In [1]:
def test():
    print("Function called!!")

### Case A: `a = test()` (call the function)
The function runs immediately and prints its message. Since `test()` has no `return` statement, it returns `None`, so `a` is `None`.

In [2]:
a = test()
print("Value stored in a:", a)

Function called!!
Value stored in a: None


### Case B: `a = test` (no parentheses)
No call happens here. `a` now just holds a **reference** to the same function, and we run it later with `a()`.

In [3]:
a = test
a()

Function called!!


## 2. What is a generator?

A **generator** is a special kind of function that produces a **sequence of values one at a time**, instead of building the whole sequence in memory at once.

**Key points**
- It is defined like a normal function (`def ...`).
- It contains the `yield` statement instead of (or along with) `return`.
- Calling a generator function does **not** run its body. It returns a **generator object** (an iterator).
- Values are produced **lazily**, only when asked for (via `next()` or a `for` loop).
- This lets a function behave like an iterator, so it works naturally in loops.

| | `return` | `yield` |
|---|---|---|
| Ends the function? | Yes | No, it **pauses** it |
| Remembers local state? | No | Yes, it resumes where it left off |
| Number of values | One | Many (one per `yield`) |

## 3. Your first generator

In [4]:
def demo():
    yield 10
    yield 20
    yield 30

Calling `demo()` doesn't execute the body. It returns a generator object.

In [5]:
d = demo()
print(d)
print(type(d))

<generator object demo at 0x7f52d06935e0>
<class 'generator'>


Use `next()` to pull values out **one at a time**. Each call runs the function until the next `yield`.

In [6]:
d = demo()
print(next(d))  # 10
print(next(d))  # 20
print(next(d))  # 30

10
20
30


### What happens when the values run out?
Once the generator is exhausted, calling `next()` raises **`StopIteration`**. A `for` loop handles this automatically behind the scenes.

In [7]:
try:
    print(next(d))
except StopIteration:
    print("StopIteration: the generator is exhausted")

StopIteration: the generator is exhausted


> **Note:** A generator can only be consumed **once**. To start over, create a new generator object (`d = demo()`).

## 4. How a generator pauses and resumes

The `print` statements below show the exact order of execution. The function **freezes** at each `yield` and continues from that same line on the next `next()` call.

In [8]:
def countdown():
    print("Start")
    yield 3
    print("Resumed after 3")
    yield 2
    print("Resumed after 2")
    yield 1
    print("Done")

c = countdown()
print("Generator created, nothing printed yet\n")

print("->", next(c))
print("->", next(c))
print("->", next(c))

Generator created, nothing printed yet

Start
-> 3
Resumed after 3
-> 2
Resumed after 2
-> 1


## 5. Generators with `for` loops

A `for` loop calls `next()` for you and stops cleanly when `StopIteration` is raised.

In [9]:
for value in demo():
    print(value)

10
20
30


## 6. Generator that wraps a list

A generator can loop over any iterable and `yield` its items one by one.

In [10]:
def gen_fun(lst):
    for item in lst:
        yield item

numbers = [123, 345, 7, 65857, 98, 456, 63, 4]
g = gen_fun(numbers)

print(next(g))
print(next(g))

123
345


The remaining values are still waiting. `list(g)` consumes **everything left**:

In [11]:
print(list(g))

[7, 65857, 98, 456, 63, 4]


## 7. Comprehensions recap: list, set, generator expression

Comprehensions are a compact way to build collections. **Only the brackets change** what you get:

| Syntax | Result |
|---|---|
| `[expr for x in iterable]` | **list** comprehension |
| `{expr for x in iterable}` | **set** comprehension |
| `(expr for x in iterable)` | **generator expression** (lazy iterator) |

In [12]:
l1 = [1, 23, 34, 56, 67, 89]

# List comprehension: multiply each element by 2
print([i * 2 for i in l1])

[2, 46, 68, 112, 134, 178]


In [13]:
# Set comprehension: unordered, no duplicates
print({i * 2 for i in l1})

{2, 68, 134, 46, 112, 178}


In [14]:
# List comprehension with a condition: keep only even numbers
print([i for i in l1 if i % 2 == 0])

[34, 56]


### Generator expression
Replacing `[]` with `()` turns a list comprehension into a **generator**. It is an iterator, so we read it with `next()`.

In [15]:
list1 = [123, 345, 7, 65857, 98, 456, 63, 4]
g = (i * 2 for i in list1)

print(g)
print(next(g))
print(next(g))

<generator object <genexpr> at 0x7f52d04d85f0>
246
690


## 8. Iterable vs Iterator, and `iter()`

- **Iterable**: an object you can loop over (list, tuple, string, set, dict...). It has `__iter__()`.
- **Iterator**: an object that produces items one at a time. It has `__next__()` and remembers its position.
- Every **generator is an iterator**.

A list is iterable but is **not** an iterator, so `next(list1)` fails. The built-in `iter()` converts any iterable into an iterator.

In [16]:
list1 = [123, 345, 7, 65857, 98, 456, 63, 4]

try:
    next(list1)
except TypeError as e:
    print("TypeError:", e)

TypeError: 'list' object is not an iterator


In [17]:
it = iter(list1)   # iterable -> iterator

print(next(it))
print(next(it))
print(next(it))

123
345
7


## 9. Practice problems

### Problem 1
Write a generator function `gen_seq()` that generates the numbers from **101 to 200**.

> **Fix from my first attempt:** I used `range(101, 202)`, which goes up to 201. `range()` excludes the stop value, so the correct call is `range(101, 201)`.

In [18]:
def gen_seq():
    for i in range(101, 201):
        yield i

g = gen_seq()
print(next(g))   # first value: 101
print(next(g))   # second value: 102

remaining = list(g)
print("Last value:", remaining[-1])      # 200
print("Total values left:", len(remaining))

101
102
Last value: 200
Total values left: 98


### Problem 2
Same idea, but take **`start`** and **`end`** as parameters and use a **`while` loop** instead of `range()`.

The sequence runs from `start` up to (but not including) `end`.

In [19]:
def gen_seq(start, end):
    i = start
    while i < end:
        yield i
        i += 1

# Fixed values so the notebook runs top to bottom without user input.
# To make it interactive, use:
#   start = int(input("Enter start = "))
#   end   = int(input("Enter end = "))
start, end = 2, 10

g = gen_seq(start, end)
print(next(g))          # 2
print(next(g))          # 3
print(list(g))          # the rest: [4, 5, 6, 7, 8, 9]

2
3
[4, 5, 6, 7, 8, 9]


## 10. Why generators? Memory efficiency

A list stores **every** item in memory at once. A generator stores only its current state and computes each value on demand, which makes a big difference for large data.

In [20]:
import sys

big_list = [i for i in range(1_000_000)]   # list: all 1,000,000 values in memory
big_gen  = (i for i in range(1_000_000))   # generator: values computed on demand

print("List size     :", sys.getsizeof(big_list), "bytes")
print("Generator size:", sys.getsizeof(big_gen), "bytes")

List size     : 8448728 bytes
Generator size: 192 bytes


Generators can even be **infinite**, which is impossible with a list:

In [21]:
def count_forever(start=0):
    n = start
    while True:
        yield n
        n += 1

counter = count_forever(100)
print(next(counter), next(counter), next(counter))

100 101 102


## 11. Summary / cheat sheet

- A **generator function** uses `yield`. Calling it returns a **generator object** and does not run the body.
- `next(gen)` runs the body until the next `yield`, returns that value, then **pauses** (local variables are preserved).
- When there are no more values, `StopIteration` is raised. `for` loops handle this automatically.
- A generator can be consumed **only once**.
- **Generator expression**: `(x * 2 for x in data)` is the short form of a generator function.
- **Iterable** (list, str, ...) vs **Iterator** (has `__next__`). Use `iter(iterable)` to get an iterator.
- Generators are **lazy and memory-efficient**, good for large or infinite sequences.

### Next topics to explore
- `send()`, `close()` and `throw()` on generators
- `yield from` (delegating to another generator)
- Building data pipelines by chaining generators
- Reading large files line by line with generators